### Getting Started:
- Make sure you are outside the sam2 repository, then run `pip install -r requirements.txt`

In [1]:
import torch
import numpy as np
from PIL import Image
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
import matplotlib.pyplot as plt
import cv2
import pandas as pd

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":
    # use bfloat16 for the entire notebook
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
elif device.type == "mps":
    print(
        "\nSupport for MPS devices is preliminary. SAM 2 is trained with CUDA and might "
        "give numerically different outputs and sometimes degraded performance on MPS. "
        "See e.g. https://github.com/pytorch/pytorch/issues/84936 for a discussion."
    )

In [3]:
def show_anns(anns, borders=True, path='example_image.png'):
    """
    NOTE: This function assumes you are saving to a folder called 'output_masks' 
    in the parent dir.
    """
    if len(anns) == 0:
        return
    sorted_anns = sorted(anns, key=(lambda x: x['area']), reverse=True)
    ax = plt.gca()
    ax.set_autoscale_on(False)
 
    img = np.ones((sorted_anns[0]['segmentation'].shape[0], sorted_anns[0]['segmentation'].shape[1], 4))
    # print("Initial canvas:", np.array(img))
    
    # Set default pixel transparency to ... (0 for transparent, 1 for full)
    img[:,:,3] = 1 

    # At this point, sorted_anns is a list of the segmented masks SAM2 found in the data
    for ann in sorted_anns:
        # For every area in the segmentation, get the mask at anns['segmentation']...
        m = ann['segmentation']
        
        # Use a color (that is NOT white)
        color_mask = np.concatenate([np.random.uniform(low=0.1, high=1, size=(3)), [1]])
        img[m] = color_mask 
        # and plot the colors
        if borders:
            contours, _ = cv2.findContours(m.astype(np.uint8),cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE) 
            # Try to smooth contours
            contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
            cv2.drawContours(img, contours, -1, (0,0,1,0.4), thickness=1) 
 

    ax.imshow(img)
    # print("Anns mask", sorted_anns[0]['segmentation'].shape)
    # print("Image as array:", np.array(img))

    

    plt.imsave(f"../output_masks/{path}", img)

In [4]:
def noisify_image(image:np.array, noise:str="random", threshold=0.1):
    # add noise to image
    # snow is generally greyscale, so noise values should be in that range
        # uint8 dtype limits cv2 randn to [0, 255]
    noise_mask = np.zeros(shape=(image.shape[0], image.shape[1], 3), dtype=np.uint8)

    if noise == "gaussian":
        # apply gaussian noise
        cv2.randn(noise_mask, mean=(128, 128, 128), stddev=(40, 40, 40))
        noise_mask = (noise_mask * 0.5).astype(np.uint8) # dilute the noise so that its application to the iamge is more realistic
        image = np.add(image, noise_mask)

    # add more noises methods here...
    elif noise == "random":
        for i in range(image.shape[0]):
            for j in range(image.shape[1]):
                if np.random.random() <= threshold:
                    image[i][j] = (np.random.rand(3) * 255).astype(np.uint32)
        

    return image

In [5]:
checkpoint = "./checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
mask_generator = SAM2AutomaticMaskGenerator(build_sam2(model_cfg, checkpoint, device=device, apply_postprocessing=False))

In [ ]:
image=Image.open("../0001TP_009240.png")
image=np.array(image.convert("RGB"))

print("Before Noise:")
plt.figure(figsize=(20, 20))
plt.imshow(image)
plt.axis('off')
plt.show()

# thresholds 0.3 and above are a bit brutal - like a snow day snowstorm.
image = noisify_image(image, noise="random", threshold=0.1)

print("After Noise:")
plt.figure(figsize=(20, 20))
plt.imshow(image)
plt.axis('off')
plt.show()

In [ ]:
masks = mask_generator.generate(image)

plt.figure(figsize=(20,20))
plt.imshow(image)
show_anns(masks)
plt.axis('off')
plt.show() 

In [ ]:
# Getting individual masks from SAM2

# Example image - clean, no noise
image=Image.open("../0001TP_009240.png")
image=np.array(image.convert("RGB"))

masks = mask_generator.generate(image)
for key in masks[0]: # assume sam2 has 1 or more masks
    print(key, ":", masks[0][key])

print("--------------------------------------------")
print("SAM2 Output Mask example shape:", masks[0]['segmentation'].shape)
print("Input image shape:", image.shape)

print("--------------------------------------------")
first_component = image
for i in range(first_component.shape[0]):
    for j in range(first_component.shape[1]):
        if not masks[0]['segmentation'][i][j]:
            first_component[i][j] = [0,0,0] 

plt.figure(figsize=(20, 20))
plt.imshow(first_component)
plt.axis('off')
plt.show()

## Common Bugs:
- "Torch is not compiled with Cuda" - uninstall torch, torchvision, torchaudio (`pip uninstall torch torchvision torchaudio`) and install the newest versions of each w/ Cuda (command available on Pytorch website). 
    - ATM this is:
     ```pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124```
- "Not implemented error" - this issue occurs when the device is Cuda but the Cuda version is not compatible with the GPU. Again, same fix as above (essentially update your Cuda version)
- "Commit denied by prereceive hook" - this issue occurs when the notebook is run & committed with all output. The output images in the notebook cause it to grow beyond git's acceptable size (100-120 mb). To fix, clear all output and restart kernel before committing.

In [9]:
import os
import random

dataset_path = "../download_dataset"
output_path = "../output_masks/val"

val_dir = os.path.join(dataset_path, 'val')

In [10]:
k = 0 # to see effects, set k > 0
val_files = os.listdir(val_dir)
noise="random" # Disable by setting to None
show_img = True

for file in random.choices(val_files, k=k):
    try:
        src = os.path.join(val_dir, file)
        
        # Get original image
        image=Image.open(src)
        image=np.array(image.convert("RGB"))
        
        if show_img:
            plt.figure(figsize=(20, 20))
            plt.imshow(image)
            plt.axis('off')
            plt.show()
            print("Original Image")

        if noise:
            image = noisify_image(image)
            
            if show_img:
                print("After Noise:")
                plt.figure(figsize=(20, 20))
                plt.imshow(image)
                plt.axis('off')
                plt.show()

        # Show SAM2 Output
        masks = mask_generator.generate(image)
        plt.figure(figsize=(20,20))
        plt.imshow(image)
        show_anns(masks, path=f"val/{file}")
        plt.axis('off')
        plt.show() 
        print("SAM2 Segmented Output")

        # Show Ground Truth
        file_labelled = file[:-4] + "_L" + file[-4:] # labelled files have a '_L' in them
        src_labelled = os.path.join(dataset_path, 'val_labels')
        src_labelled = os.path.join(src_labelled, file_labelled) # Get the related labeled image filepath
        image=Image.open(src_labelled)
        image=np.array(image.convert("RGB"))
        plt.figure(figsize=(20, 20))
        plt.imshow(image)
        plt.axis('off')
        plt.show()
        print("Ground Truth")
    except:
        print("An image could not be found. Moving on...")

In [ ]:
# Testing SAM2's resilience to noise
max_times_ten=0 # set to 5 to see full range

found_image = False
while not found_image:
    try:
        file = random.choice(val_files)
        src = os.path.join(val_dir, file)    
        # Get original image
        image=Image.open(src)
        image=np.array(image.convert("RGB"))
        found_image = True
    except:
        print("Could not find an image. Trying another ...")

# Test SAM2's resilience to noise
for i in range(1, max_times_ten+1):
    t = i / 10
    image = noisify_image(image, threshold=t)
    
    print(f"Noisy Image at threshold {i / 10}:")
    plt.figure(figsize=(20, 20))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

    # Show SAM2 Output
    masks = mask_generator.generate(image)
    plt.figure(figsize=(20,20))
    plt.imshow(image)
    show_anns(masks, path=f"val/threshold_{t}_{file}")
    plt.axis('off')
    plt.show() 
    print("SAM2 Segmented Output")

# Show Ground Truth
try:
    file_labelled = file[:-4] + "_L" + file[-4:] # labelled files have a '_L' in them
    src_labelled = os.path.join(dataset_path, 'val_labels')
    src_labelled = os.path.join(src_labelled, file_labelled) # Get the related labeled image filepath
    image=Image.open(src_labelled)
    image=np.array(image.convert("RGB"))
    plt.figure(figsize=(20, 20))
    plt.imshow(image)
    plt.axis('off')
    plt.show()
    print("Ground Truth")
except:
    print("Could not find ground truth. Moving on...")

<h2>YOLO & SAM2</h2>

In [ ]:
%pip install ultralytics

In [ ]:
table = pd.read_csv("../download_dataset/class_dict.csv")
# print(table.head())

# dict based on english labels
class_dict = {row["name"].lower() : [row["r"] / 255, row["g"] / 255, row["b"] / 255] for _, row in table.iterrows()}
color_labels = [class_dict[key] for key in class_dict]

print(class_dict)
label_map = {
    'person':'pedestrian'
}

def show_mask(mask, ax, label=None, index=-1):
    # edit so that color aligns w/ camvid class dict

    if label:
        if label in class_dict:
            color = np.concatenate([np.array(class_dict[label]), np.array([1])], axis=0)

        else:
            if label in label_map:
                label = label_map[label]
                if label in class_dict:
                    color = np.concatenate([np.array(class_dict[label]), np.array([1])], axis=0)
                else:
                    print(label)
                    color = np.concatenate([np.random.random(3), np.array([1])], axis=0)
            else:
                print(label)
                color = np.concatenate([np.random.random(3), np.array([1])], axis=0)
    else:
        if index > -1 and index < len(color_labels):
            color = np.concatenate([np.array(color_labels[index]), np.array([1])], axis=0)
        else:
            color = np.concatenate([np.random.random(3), np.array([1])], axis=0)
    
    
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)
    
def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)   
    
def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0,0,0,0), lw=2))    

In [56]:
# bbox = np.reshape(bbox, shape=(-1,1)).shape

def SAM2_segment_box(image, predictor, bbox, labels=None):
    predictor.set_image(image=image)
    plt.figure(figsize=(10, 10))
    plt.imshow(image)

    n = len(bbox)
    for i in range(n):
        obj_detected, label = bbox[i], labels[i]

        input_box = np.array(obj_detected)

        masks, _, _ = predictor.predict(
            point_coords=None,
            point_labels=None,
            box=obj_detected,
            multimask_output=False
        )
        
        show_mask(masks[0], plt.gca(), label=label)
        show_box(input_box, plt.gca())
    plt.axis('off')
    plt.show()

In [57]:
from ultralytics import YOLO

img_path = "../0001TP_009240.png"

yolo_model = YOLO('yolov8n.pt')

sam2_checkpoint = "./checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device=device)
predictor = SAM2ImagePredictor(sam2_model)

In [ ]:
results = yolo_model.predict(source=img_path, conf=0.25)
names = results[0].names

for result in results:
    boxes = result.boxes
    bbox = boxes.xyxy.tolist() # bounding boxes AKA areas where YOLO found an object
    
    # print(boxes.cls)
    # print(result.names)

    image=Image.open(img_path)
    image=np.array(image.convert("RGB"))

    plt.figure(figsize=(20, 20))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

    SAM2_segment_box(image=image, predictor=predictor, bbox=bbox, labels = [names[index] for index in boxes.cls.tolist()])